## 1. **Required prior knowledge**

   - Arduino and QRoboticsCenter software setup 
   - Introduction to Bayesian filtering 
   - Introduction to Kalman filtering

## 2. **Assignment context:** 

Every Bayesian filter algorithm that we have use in this course consist of a **prediction** and an **update step** as shown in the provided lecture notes and the figure below: 


![image]( images/Steps.png)

In this assignment we will learn how to define the prediction step for a mobile robot

**General prediction step**

  1. Use process model to predict state at the next time step:  $\bar{\mathbf x} = f(\mathbf{x},\mathbf{u})$
  
  
  2. Adjust belief to account for the uncertainty in prediction  
    
 <br>

**Linear Kalman filter prediction step**

$\begin{array}{|l|l|}
    \hline
 &\text{Multivariate Kalman filter} \\
\hline
\text{Process model equation:} &\bar{\mathbf x} = \mathbf{Fx} + \mathbf{Bu}\\
\text{Prediction uncertainty evolution:}&\bar{\mathbf P} = \mathbf{FPF}^\mathsf T + \mathbf Q \\
\hline
\end{array}$

given that:

 - $\mathbf x,\, \mathbf P$ are the state mean and covariance.

 - $\mathbf F$ is the **state transition function**. When multiplied by $\bf x$ it computes the prior. 

 - $\mathbf Q$ is the process covariance. 

 - $\mathbf B$ and $\mathbf u$ are the **input transition matrix** and the vector of input values respectively .

## 3. **Assignment goals**

This assignment is to determine a robot motion process model for two scenarios:

* **1.** The position based on the speed. Assume 1D motion of the robot platform.
* **2.** The position (2D + orientation!) based on the IMU measurements. For this you will need to calibrate the IMU position and orientation.

Both process models will consist of:   

 - the **process model** matrixes  ($\mathbf{F},\mathbf{B}$) and, 
 - corresponding uncertainty ($\mathbf Q$).

Which of these dead reckoning approaches is more accurate? Why? 

 
## 4. **Submission format**
 
You need to communicate your result using an jupyter-notebook file  and submit it in using a Toledo assignment. The applied code and results should be clearly explained in a condense way.


## APPENDIX

### **How to collect mobile robot sensor data**

  Read the [Jupyter Notebook](https://gitlab.kuleuven.be/fet-edu-em/mobile-robotics-t4smr2/student_material/assignments/-/blob/main/Assignment1_software_setup/Assignment_software-setup.ipynb?ref_type=heads) from the Software setup assignment folder.

### **How to adapt motor control values**

  Change the code in robot.cpp after the following line:
  ```C
    if (controlEnabled())
  ```

### **How to change the values displayed in QRoboticsCenter**

  Change the code in robot.cpp that writes values from ```out[0]``` to ```out[11] ```.

### **How to read a csv file with Python**
  
  ```Python
  import pandas as pd 
  from pandas import read_csv
  df = read_csv('filename.csv', header=1)
  ```

### **What is a good DC motor model**

1. The electrical part: 

$$ V_a= L \frac{dI_a}{dt} + R_a I_a + K_e \omega $$

2. The mechanical part:  

$$ K_T I_a = J \frac{d \omega}{dt} + c \omega $$

where the following variables are measured:

 - $I_a$ the motor current;
 - $V_a$ the motor voltage; 
 - $\omega$ the motor rotation speed; 
 
and the following parameters are unknown: 

 - L the motor induction; 
 - $R_a$ the motor resistance;
 - $K_e$ the counter EMK constant;
 - $K_t$ the motor torque constant; 
 - J is the motor inertia = $J_m + \frac{1}{n^2} J_w$;
 - n is the gear ratio between motor to wheel $= \frac{n_m}{n_w}$
 - c is the motor viscous friction coefficient. 
 
A lot of the unknown parameters can be derived from the [motor datasheet](https://users.obs.carnegiescience.edu/crane/pfs/man/Electronics/MicroMo2224.pdf):

 - L = 203 µH
 - $R_a$ = 8.77 $\Omega$
 - $K_e$ =  1.52 $mV/rpm$ = 0.0145 V/(rad/sec)
 - $K_t$ = 14.5 mNm/A = 0.0145 Nm/A 
 - $J_m$ = 2.7 gcm²
 - n = 256/9

### **IMU Datasheet**

Check the [accelerometer datasheet](https://www.hestore.hu/prod_getfile.php?id=8301) for its specifications.

### **How to do linear regression**

Linear regression can be done in Python using for example the [statsmodel toolbox](https://www.statsmodels.org/stable/regression.html) or [sci-kit Learn](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html). Example code is provided below.

In [12]:
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm

USE_SKLEARN = True
USE_STATSMODELS = True

# 1. Generate a dataset with noise around the equation: y = 2.5*x_1 - 1.8*x_2
np.random.seed(42)
n_samples = 100
x1 = np.random.uniform(0, 10, n_samples)
x2 = np.random.uniform(0, 10, n_samples)
a_true, b_true = 2.5, -1.8
noise = np.random.normal(loc=0.0, scale=1.5, size=n_samples)
y = a_true * x1 + b_true * x2 + noise
X = np.column_stack((x1, x2))

# 2. Fit coefficients a and b to the dataset
if USE_SKLEARN:
    model = LinearRegression(fit_intercept=False).fit(X, y)
    a_fit, b_fit = model.coef_
elif USE_STATSMODELS:
    model = sm.OLS(y, X).fit()
    print(model.summary())
    a_fit, b_fit = model.params
else:
    print("select a library")
    exit()

print(f"True coefficients:   a = {a_true}, b = {b_true}")
print(f"Fitted coefficients: a = {a_fit:.2f}, b = {b_fit:.2f}")

# 3. Plot the results interactively
x1_range = np.linspace(0, 10, 20)
x2_range = np.linspace(0, 10, 20)
x1_grid, x2_grid = np.meshgrid(x1_range, x2_range)
y_grid = a_fit * x1_grid + b_fit * x2_grid
fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=x1, y=x2, z=y,
    mode='markers',
    marker=dict(size=4, color='blue', opacity=0.7),
    name='Data points'
))
fig.add_trace(go.Surface(
    x=x1_range, y=x2_range, z=y_grid,
    colorscale='Reds', opacity=0.5,
    name='Fitted Plane', showscale=False
))
fig.update_layout(
    scene=dict(xaxis_title='x1', yaxis_title='x2', zaxis_title='y'),
    margin=dict(l=0, r=0, b=0, t=30),
    title='Interactive 3D Linear Regression'
)
fig.show()

True coefficients:   a = 2.5, b = -1.8
Fitted coefficients: a = 2.44, b = -1.70
